# Layup Visualize Demo Notebook
Once you have your lovely set of input orbits, why not take a moment to appreciate how they look in space? The pupose of this notebook is to demonstrate a simple way to display the Layup Visualize orbital drawing function (as an ellipse/cone) within Jupyter, without needing to run the full Dash app. The basic driver behind all of this is that we convert any orbit format into a set of classical conic section elements, namely:

- L (Semilatus rectum, in au)
- e (Eccentricity)
- i (Inclination, in degrees)
- $\omega$ (Argument of perihelion, in degrees)
- $\Omega$ (Longitude of ascending node, in degrees) 

From these we can calculate a true anomaly $\nu$ and describe any conic section, from circular to elliptical to parabolic to hyperbolic orbits.

<p align="center">
    <img src="./images/conicsections.png" alt="conic sections" width="500"/>
    <img src="./images/orbitellipse.png" alt="orbit ellipse" width="800"/>
</p>

In [1]:
import numpy as np

from layup.orbit_maths import (
    conic_lines_from_classical_conic,
    build_ephem_and_mus,
    build_planet_lines_cache,
    prepopulate_orbit_variants,
)
from layup.dash_ui import plotly_2D, plotly_3D

## 1) Create (or load) some orbits
Below we build a few toy orbits directly with our classical conic elements. In your real workflow, you’d typically get a `ClassicalConic` object via our existing converters in `orbit_maths.py`

In [2]:
# --- toy dataset: 5 example orbits ---
rng = np.random.default_rng(42)
N = 5
dtype = [
    ("ObjID", "U32"),
    ("epochMJD_TDB", "f8"),
    ("a", "f8"),
    ("e", "f8"),
    ("inc", "f8"),
    ("argPeri", "f8"),
    ("node", "f8"),
    ("ma", "f8"),
]

orbits = np.zeros(N, dtype=dtype)
orbits["ObjID"] = [f"Obj-{i:02d}" for i in range(N)]
orbits["epochMJD_TDB"] = float(61000.0)
orbits["a"] = rng.uniform(1.5, 12.0, size=N)
orbits["e"] = np.clip(
    rng.normal(0.2, 0.15, size=N), 0.01, 0.85
)  # <-- keep it elliptical for the sake of discussion, but could be hyperbolic
orbits["inc"] = rng.uniform(0, 60, size=N)
orbits["argPeri"] = rng.uniform(0, 360.0, size=N)
orbits["node"] = rng.uniform(0, 360.0, size=N)
orbits["ma"] = rng.uniform(0, 360, size=N)

print(orbits)

[('Obj-00', 61000.,  9.62653851, 0.01      , 22.24788145,  81.80593984, 272.91158643,  70.06993483)
 ('Obj-01', 61000.,  6.10822362, 0.21917606, 55.60589933, 199.65052333, 127.62934853, 168.01956134)
 ('Obj-02', 61000., 10.51527816, 0.15256361, 38.6319072 ,  22.9742122 , 349.45128878,  15.76935568)
 ('Obj-03', 61000.,  8.82236431, 0.19747983, 49.3656968 , 297.94722192, 321.52360368,  55.54421714)
 ('Obj-04', 61000.,  2.48886215, 0.07204341, 26.60485193, 227.39918368, 280.21805895, 245.89762317)]


## 2) Generate a cache of all ellipse/hyperbola lines
Under the hood, Layup Visualize will create a cache in all four combinations of barycentric+heliocentric && equatorial+ecliptic, regardless of your input specification via `prepopulate_orbit_variants` (we just use your input to tell us where to start for conversion). This allows us to easiy toggle between any combination you want

In [3]:
conic_cache, lines_cache, sunpos_cache, pos_cache = prepopulate_orbit_variants(
    orbits,
    orbit_format="KEP",
    input_plane="ecliptic",
    input_origin="heliocentric",
)

## 3) Get the heliocentric ecliptic frame orbits

By default in the Dash app we render our orbits, regardless of input, as heliocentric ecliptic framed. In the app you can toggle between heliocentric <-> barycentric origins and ecliptic <-> equatorial reference planes no matter what input you gave via the buttons. For now though, let's specify explicitly the output we want as `key`

`conic_lines_from_classical_conic` then returns an array shaped `(N, n_points, 3)` with XYZ points in au for your N objects

In [4]:
key = ("helio", "ecl")
conic = conic_cache[key]
orbit_pos = pos_cache[key]
sun_xyz = sunpos_cache[key]

lines = conic_lines_from_classical_conic(conic, n_points=400)
lines.shape

(5, 400, 3)

## 4) Plot objects (2D and 3D)

`plotly_2D` and `plotly_3D` can now produce some nice figures of our orbits. Here we'll first render all of our objects in our set in 2D. `plotly_2d` takes an argument `plots` (plural) which can be one of XY / XZ / YZ. This produces side by side of any of: a bird's eye (top-down, or XY plane) view, or a side (edge-on, or XZ / YZ plane) view of all our orbits, as well as the Sun. We note that XY is always returned with equal aspect ratios, as that is the reference plane, and we don't want to distort the orbit shapes. The scatter markers represent where each input object was at the reference epoch specified in the input file

Try moving your mouse over each plot to play around with the interactive aspect! Hover over an orbit to get a tooltip containing some orbital elements of that object, pan and squash/stretch the axes by clicking and dragging your mouse along them, or zoom in and out of the figure. In the top right, you can reset the figure to its initial position, as well as save the figure as a .png if you like

In [5]:
fig2d = plotly_2D(lines, conic, orbit_pos=orbit_pos, plot_sun=True, panels=("XY", "XZ"), return_fig=True)
fig2d

2D plots can also be single panelled if the user prefers. `plotly_2d` takes an argument for `panel` (singular this time) that can be one of XY / XZ / YZ. All other parts of plotting stay the same!

In [6]:
fig2d = plotly_2D(lines, conic, orbit_pos=orbit_pos, plot_sun=True, panel="XY", return_fig=True)
fig2d

We can also add some planets on top quite easily by just constructing them in a very similar way to our orbits. The only difference is we need to make use of Assist to get accurate Sun + Planet positions based on the users bootstrapped auxiliary files

In [7]:
ephem, _, _ = build_ephem_and_mus(None)
epochJD_center = float(61000.0 + 2400000.5)

planet_names = ["Mercury", "Venus", "Earth", "Mars", "Jupiter", "Saturn"]
planet_lines_cache, planet_id = build_planet_lines_cache(
    ephem,
    epochJD_center,
    planet_names=planet_names,
    n_points=900,
)

fig2d = plotly_2D(
    lines,
    conic,
    orbit_pos=orbit_pos,
    planet_lines=planet_lines_cache[key],
    planet_id=planet_id,
    plot_sun=True,
    panels=("XY", "XZ"),
    return_fig=True,
)
fig2d

3D plots are equally as easy, just call `plotly_3D` instead! Again, we plot in heliocentric ecliptic by default, and now we overplot the ecliptic plane in red to help you orient yourself better. On the Dash app version you can control the transparency of this (including turning it off entirely - if you want to do this here, just change `show_plane` to False), and convert it to the equatorial plane via a toggle button. Planets can be added in the exact same way as before. Play around and try hovering over orbit lines again to see how our toy objects look in 3D this time. Try lining up the XY and XZ planes as in the 2D case above - do they match? 

In [8]:
fig3d = plotly_3D(
    lines,
    conic,
    orbit_pos=orbit_pos,
    planet_lines=planet_lines_cache[key],
    planet_id=planet_id,
    plot_sun=True,
    show_plane=True,
    return_fig=True,
)
fig3d

## 5) Parabolic & hyperbolic orbits

Parabolic and hyperbolic orbits are naturally handelled by Layup Visualize in the exact same way as any other regular orbit. The only difference is we set a maximum distance that we draw the parabolae/hyperbolae out to, which the user can alter if they want to see more or less of the trajectory

In [9]:
# --- toy dataset: 5 example hyperbolic orbits ---
rng = np.random.default_rng(24601)
N = 5
dtype = [
    ("ObjID", "U32"),
    ("epochMJD_TDB", "f8"),
    ("q", "f8"),
    ("e", "f8"),
    ("inc", "f8"),
    ("argPeri", "f8"),
    ("node", "f8"),
    ("t_p_MJD_TDB", "f8"),
]

orbits = np.zeros(N, dtype=dtype)
orbits["ObjID"] = [f"Obj-{i:02d}" for i in range(N)]
orbits["epochMJD_TDB"] = float(61000.0)
orbits["q"] = rng.uniform(1.5, 12.0, size=N)
orbits["e"] = np.clip(rng.normal(5.0, 2.5, size=N), 1.0, 10.0)  # <-- keep it hyperbolic now
orbits["inc"] = rng.uniform(0, 60, size=N)
orbits["argPeri"] = rng.uniform(0, 360.0, size=N)
orbits["node"] = rng.uniform(0, 360.0, size=N)
orbits["t_p_MJD_TDB"] = float(58000.0)

# --- build lines again, but set a maximum render distance of 60 au ---
conic_cache, lines_cache, sunpos_cache, pos_cache = prepopulate_orbit_variants(
    orbits,
    orbit_format="COM",
    input_plane="ecliptic",
    input_origin="heliocentric",
)

planet_names = ["Mercury", "Venus", "Earth", "Mars", "Jupiter", "Saturn", "Uranus", "Neptune"]
planet_lines_cache, planet_id = build_planet_lines_cache(
    ephem,
    epochJD_center,
    planet_names=planet_names,
    n_points=900,
)

key = ("helio", "ecl")
conic = conic_cache[key]
orbit_pos = pos_cache[key]
sun_xyz = sunpos_cache[key]

lines = conic_lines_from_classical_conic(conic, n_points=400, r_max=60)

# --- draw 2d plot ---
fig2d = plotly_2D(
    lines,
    conic,
    orbit_pos=orbit_pos,
    planet_lines=planet_lines_cache[("helio", "ecl")],
    planet_id=planet_id,
    plot_sun=True,
    panels=("XY", "XZ"),
    return_fig=True,
)
fig2d

Or again, equivalently in 3D

In [10]:
fig3d = plotly_3D(
    lines,
    conic,
    orbit_pos=orbit_pos,
    planet_lines=planet_lines_cache[key],
    planet_id=planet_id,
    plot_sun=True,
    show_plane=True,
    return_fig=True,
)
fig3d

## 6) (Optional) Altering the look of the figure
Layup Visualize supplies a default figure look via some internal CSS styling. Some users may have their own artistic idea in mind however for how they want orbits to look, so we give an example below on how you can use Plotly's inbuilt methods to update and decorate your plots to your hearts content. Have a look at [the Plotly documentation](https://plotly.com/python/reference/) for more information on what you can change and how to do it

In [11]:
fig2d = plotly_2D(
    lines,
    conic,
    orbit_pos=orbit_pos,
    planet_lines=planet_lines_cache[("helio", "ecl")],
    planet_id=planet_id,
    plot_sun=True,
    panels=("XY", "XZ"),
    return_fig=True,
)

# --- layout / background ---
fig2d.update_layout(
    plot_bgcolor="grey",
    paper_bgcolor="grey",
)

# --- x-axes ---
fig2d.update_xaxes(
    title_text="X [AU]",
    showgrid=True,
    showline=True,  # <-- draws axis spines top/right, plotly only draws bottom/left by default
    mirror=True,
    linecolor="black",
    gridcolor="lightgray",
    tickfont=dict(color="black"),
    zeroline=False,
)

# --- y-axes ---
fig2d.update_yaxes(
    title_text="Y [AU]",
    showgrid=True,
    showline=True,
    mirror=True,  # <-- draws axis spines top/right, plotly only draws bottom/left by default
    linecolor="black",
    gridcolor="lightgray",
    tickfont=dict(color="black"),
    zeroline=False,
)

# --- style orbit lines + markers ---
fig2d.for_each_trace(
    lambda tr: (
        tr.update(line=dict(color="red"))
        if tr.mode == "lines"
        and (
            tr.meta is None or tr.meta.get("kind") != "planet"
        )  # <-- planets have a special internal tag so you can colour input objects separately
        else None
    )
)

fig2d.update_traces(
    selector=dict(mode="markers"),
    marker=dict(size=8, color="blue"),
)


fig2d

## 7) (Optional) Saving the figure
Plotly offers the ability to save the current frame, however you choose to orient, zoom, or pan the plot, as a png file via the camera button in the top right of the output. Alternatively, we have specified `return_fig=True` in `plotly_2D`/`plotly_3D`, which means we can make use of a Plotly feature to save to any format via the following (see the [Plotly documentation](https://plotly.github.io/plotly.py-docs/generated/plotly.io.write_image.html) for more info on this method, you may need to install external image exporting engines to properly utilise this method): 

In [ ]:
fig2d.write_image("/foo/bar/baz/image.pdf")
fig3d.write_image("/foo/bar/baz/image.pdf")

# 8) (Optional) Running the Dash app in a notebook
We've been mentioning running the Dash App in the web (i.e. in a window in your browser of choice) a lot so far, but it is possible to get the full interactivity of Layup Visualize in a Jupyter notebook cell. We've made a convenient wrapper funciton, `visualize_notebook` that essentially calls the CLI functions behind the scenes with either a string path to your input file, or a structured numpy array like we've been making for our toy models here. Let's call it here on another set of toy orbits: 

In [12]:
from layup.visualize import visualize_notebook
import numpy as np

# --- toy dataset: 5 example orbits ---
rng = np.random.default_rng(8675309)
N = 5
dtype = [
    ("ObjID", "U32"),
    ("epochMJD_TDB", "f8"),
    ("a", "f8"),
    ("e", "f8"),
    ("inc", "f8"),
    ("argPeri", "f8"),
    ("node", "f8"),
    ("ma", "f8"),
    ("FORMAT", "U32"),
]

orbits = np.zeros(N, dtype=dtype)
orbits["ObjID"] = [f"Obj-{i:02d}" for i in range(N)]
orbits["epochMJD_TDB"] = float(61000.0)
orbits["a"] = rng.uniform(1.5, 12.0, size=N)
orbits["e"] = np.clip(
    rng.normal(0.2, 0.15, size=N), 0.01, 0.85
)  # <-- keep it elliptical for the sake of discussion, but could be hyperbolic
orbits["inc"] = rng.uniform(0, 60, size=N)
orbits["argPeri"] = rng.uniform(0, 360.0, size=N)
orbits["node"] = rng.uniform(0, 360.0, size=N)
orbits["ma"] = rng.uniform(0, 360, size=N)
orbits["FORMAT"] = "KEP"

visualize_notebook(orbits)

--input-origin not provided. Inferring heliocentric from input file format column FORMAT=KEP
--input-plane not provided. Inferring ecliptic for input file format column FORMAT=KEP
Requested 100 orbits, but only 5 orbits in input. Capping to 5
